# 04 - Árvore de decisão

## Objetivo

Avaliar uma árvore de decisão para prever `AGE_RANGE` e comparar a Macro F1 com a regressão logística regularizada do notebook 002. A comparação usa as mesmas variáveis e a mesma divisão estratificada; nesta experiência a preparação numérica fica com `normalize_numeric=False`, pois a escala das variáveis não é necessária para árvores. A versão 03 mantém-se como análise de sensibilidade das quatro perguntas condicionais, não como substituta da referência 002.

## Ambiente, dados e divisão fixa

Carregamos os mesmos CSV e reutilizamos `preparar_dados()` e `make_stratified_split()` do módulo comum. A divisão é feita com `test_size=0.2` e `random_state=42`, tal como nas experiências anteriores. A validação fica reservada para a avaliação; o pré-processador da pipeline aprende apenas ao ajustar em `X_train`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import sklearn

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')

sys.path.insert(0, str(repo_root))
from src.preparar_dados import make_model_pipeline, make_stratified_split, preparar_dados

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f'CSV de treino/teste não encontrados em {repo_root / "notebooks"}')

print(f'Python: {sys.version}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'Raiz do projeto: {repo_root}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X_train, X_val, y_train, y_val = make_stratified_split(
    prepared['X_train'], prepared['y_train'], test_size=0.2, random_state=42
)

assert len(X_train) == 8769 and len(X_val) == 2193, (len(X_train), len(X_val))
assert X_train.columns.equals(X_val.columns), 'As colunas de treino e validação não estão alinhadas.'
print(f'Treino: {X_train.shape}; validação: {X_val.shape}')
print(f'Classes no alvo: {sorted(y_train.unique().tolist())}')

## Modelo inicial

Começamos com uma configuração baseline única: árvore `gini`, divisão `best`, sem limite de profundidade, `min_samples_split=2`, `min_samples_leaf=1`, sem ponderação de classes e `random_state=42`. Estes parâmetros permitem que a árvore cresça bastante; a diferença entre Macro F1 de treino e validação e a dimensão da árvore ajudam a avaliar sobreajustamento. Não há normalização numérica.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_params = {
    'criterion': 'gini',
    'splitter': 'best',
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'class_weight': None,
    'random_state': 42,
}
model = DecisionTreeClassifier(**tree_params)
pipe = make_model_pipeline(model, normalize_numeric=False)

print(f'Parâmetros da árvore: {tree_params}')
print('normalize_numeric=False')
print(pipe)

## Ajuste e avaliação sem leakage

A pipeline é ajustada apenas em `X_train` e `y_train`. A chamada a `predict(X_val)` transforma a validação com os parâmetros aprendidos no treino, sem reajustar imputação ou categorias. Reportamos Macro F1 para treino e validação, métricas por classe e matriz de confusão com as 16 classes explicitamente incluídas.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

pipe.fit(X_train, y_train)
y_train_pred = pipe.predict(X_train)
y_val_pred = pipe.predict(X_val)

macro_f1_train = f1_score(y_train, y_train_pred, labels=list(range(16)), average='macro', zero_division=0)
macro_f1_val = f1_score(y_val, y_val_pred, labels=list(range(16)), average='macro', zero_division=0)
fitted_tree = pipe.named_steps['model']

print(f'Macro F1 treino: {macro_f1_train:.4f}')
print(f'Macro F1 validação: {macro_f1_val:.4f}')
print(f'Profundidade da árvore: {fitted_tree.get_depth()}')
print(f'Número de folhas: {fitted_tree.get_n_leaves()}')
print(f'Número de nós: {fitted_tree.tree_.node_count}')
print('--- classificação por classe ---')
print(classification_report(
    y_val,
    y_val_pred,
    labels=list(range(16)),
    zero_division=0,
    target_names=[str(i) for i in range(16)],
))
print('--- matriz de confusão ---')
confusion = confusion_matrix(y_val, y_val_pred, labels=list(range(16)))
display(pd.DataFrame(confusion, index=range(16), columns=range(16)))

## Comparação com a referência 002

A referência é a regressão logística regularizada do notebook 002, avaliada na mesma divisão: Macro F1 de validação **0,4125** (`C=0.1`, `class_weight=None`, `solver='lbfgs'`, `max_iter=2000`, `random_state=42`, `normalize_numeric=True`). Compare esse valor com `macro_f1_val` acima. A comparação mantém as variáveis e a partição, mas os algoritmos e a normalização diferem; uma diferença de score, por si só, não identifica a causa nem demonstra melhor desempenho fora desta validação.

## Gerar a submissão

Criamos uma pipeline nova com a mesma configuração da árvore baseline e ajustamo-la em todas as linhas rotuladas. As previsões são associadas aos `RowId` do exemplo oficial, se existir; caso contrário, reutilizamos os identificadores da submissão 001 já validada (nunca as previsões). O CSV é guardado sem índice e não substitui ficheiros existentes.

In [ ]:
final_pipe = make_model_pipeline(
    DecisionTreeClassifier(**tree_params),
    normalize_numeric=False,
)
final_pipe.fit(prepared['X_train'], prepared['y_train'])
test_predictions = final_pipe.predict(prepared['X_test']).astype(int)

example_path = repo_root / 'example_submission_10.csv'
previous_path = repo_root / 'submissions' / 'submission_001_regressao_logistica.csv'
if example_path.is_file():
    row_reference = pd.read_csv(example_path, usecols=['RowId'])
    reference_source = example_path
elif previous_path.is_file():
    row_reference = pd.read_csv(previous_path, usecols=['RowId'])
    reference_source = previous_path
else:
    raise FileNotFoundError('É necessário example_submission_10.csv ou a submissão 001 validada para preservar RowId.')

assert len(test_predictions) == len(row_reference) == 3655
submission = row_reference.copy()
submission['AGE_RANGE'] = test_predictions
assert list(submission.columns) == ['RowId', 'AGE_RANGE']
assert submission['RowId'].equals(row_reference['RowId'])
assert pd.api.types.is_integer_dtype(submission['AGE_RANGE'])
assert submission['AGE_RANGE'].between(0, 15).all()

submission_dir = repo_root / 'submissions'
submission_dir.mkdir(exist_ok=True, parents=True)
submission_path = submission_dir / 'submission_004_arvore_decisao.csv'
suffix = 1
while submission_path.exists():
    suffix += 1
    submission_path = submission_dir / f'submission_004_arvore_decisao_{suffix}.csv'
submission.to_csv(submission_path, index=False)

saved = pd.read_csv(submission_path)
assert saved.shape == (3655, 2)
assert list(saved.columns) == ['RowId', 'AGE_RANGE']
assert saved['RowId'].equals(row_reference['RowId'])
assert pd.api.types.is_integer_dtype(saved['AGE_RANGE'])
assert saved['AGE_RANGE'].between(0, 15).all()
print(f'Submissão guardada em {submission_path}')
print(f'RowId copiado de {reference_source}')
print(saved['AGE_RANGE'].value_counts().sort_index())

## Registo e conclusão

- Data: 9/11
- Macro F1 de treino: **1,0000**.
- Macro F1 de validação: **0,3628**.
- Profundidade / folhas: **29 / 2515**.
- Diferença face à referência 002 (0,4125): **−0,0497**.
- Conclusões: A diferença entre treino e validação indica sobreajustamento nesta configuração baseline. A árvore ficou abaixo da regressão logística na divisão fixa; isto não permite concluir que todas as árvores tenham desempenho inferior.
- Submissão baseline 004: `submission_004_arvore_decisao.csv`; pontuação pública no Kaggle: **0,35**.
- Próxima experiência: testar árvores limitadas num notebook separado, sem alterar este registo baseline.